# 0: Read all JSON data from ODS tables

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from datetime import datetime

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("CombineTXTData").getOrCreate()

# Read good JSON data
df_json_good = spark.table("ODS.Employee_JSON_Data")

# Read JSON error data (if you want to include corrected data)
df_json_errors = spark.table("ODS.Employee_JSON_Data_Errors")

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 39, Finished, Available, Finished, False)

# 1: Analyze Error Types

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, split, size, expr, desc, concat, lit as spark_lit

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("AnalyzeJSONErrors").getOrCreate()

# Read JSON data
df_json_good = spark.table("ODS.employee_json_data")
df_json_errors = spark.table("ODS.employee_json_data_errors")

print("="*50)
print("JSON DATA OVERVIEW")
print("="*50)

print(f"\n✅ Good JSON records: {df_json_good.count()}")
print(f"Columns: {df_json_good.columns}")
print("\nSample good records:")
display(df_json_good.limit(3).toPandas())

print("\n" + "="*50)
print("JSON ERROR ANALYSIS")
print("="*50)

print(f"\n⚠️ Error JSON records: {df_json_errors.count()}")
print(f"Columns: {df_json_errors.columns}")
print("\nSample error records:")
display(df_json_errors.limit(3).toPandas())

print("\n📊 Error Type Distribution:")

# Separate errors by type
df_missing = df_json_errors.filter(col("error_type").contains("MISSING_FIELDS"))
df_extra = df_json_errors.filter(col("error_type").contains("EXTRA_FIELDS"))
df_both = df_json_errors.filter(col("error_type") == "MISSING_FIELDS | EXTRA_FIELDS")

print(f"\nMissing fields errors: {df_missing.count()}")
print(f"Extra fields errors: {df_extra.count()}")
print(f"Both missing and extra: {df_both.count()}")

print("\n📊 Detailed Error Breakdown:")
print("\n🔍 Missing Fields Analysis:")
df_missing_grouped = df_missing.groupBy("missing_fields").count().orderBy(desc("count"))
display(df_missing_grouped.toPandas())

print("\n🔍 Extra Fields Analysis:")
df_extra_grouped = df_extra.groupBy("extra_fields").count().orderBy(desc("count"))
display(df_extra_grouped.toPandas())

print("\n📋 Individual Error Details:")
display(
    df_json_errors.select(
        "record_number",
        "error_type",
        "missing_fields",
        "extra_fields",
        "record_data"
    ).toPandas()
)

print("\n🔍 Sample Missing Fields Errors:")
display(df_missing.select("record_number", "missing_fields", "record_data").limit(5).toPandas())

print("\n🔍 Sample Extra Fields Errors:")
display(df_extra.select("record_number", "extra_fields", "record_data").limit(5).toPandas())

print("\n🔍 Sample Both Errors:")
display(df_both.select("record_number", "missing_fields", "extra_fields", "record_data").limit(5).toPandas())

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 30, Finished, Available, Finished, False)

JSON DATA OVERVIEW

✅ Good JSON records: 986
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type']

Sample good records:


SynapseWidget(Synapse.DataFrame, ee289f22-115a-4e88-8b30-1680b5b772a7)


JSON ERROR ANALYSIS

⚠️ Error JSON records: 14
Columns: ['record_number', 'expected_count', 'actual_count', 'missing_fields', 'extra_fields', 'error_type', 'record_data', 'load_timestamp', 'source_location', 'file_type']

Sample error records:


SynapseWidget(Synapse.DataFrame, ad4655d1-5c80-4b63-ac98-acf5b6c2157e)

...

SynapseWidget(Synapse.DataFrame, 17e0b5b7-44bf-4243-9f31-1b9b04ec5654)


🔍 Sample Extra Fields Errors:


SynapseWidget(Synapse.DataFrame, defdb448-0ceb-409c-abaa-ade5fafc3330)


🔍 Sample Both Errors:


SynapseWidget(Synapse.DataFrame, 2a2a879e-6a24-4300-a573-20b66024269a)

### Issue 1: We can notice that all the rows has both missing and extra columns
### Issue 2: We can notice that there are a difference in the namings of the columns of the salary and dept to compensation and department

### Extra Note: We can notice that these columns has issues are full_name, f_name, l_name and these are mapped to name
### Extra Note: We can notice that this column has an issue salary and it is mapped to compensation
### Extra Note: We can notice that this column has an issue dept and it is mapped to department

# 2: Investigate Missing Columns Errors

In [13]:
print("="*50)
print("MISSING FIELDS ERRORS")
print("="*50)

# Show sample of missing field errors
print("\n🔍 Sample Missing Field Errors:")
display(df_missing.limit(10).toPandas())

# Analyze missing field patterns
print("\n📊 Missing Field Patterns:")
df_missing.select(
    col("expected_count"),
    col("actual_count"),
    col("missing_fields"),
    col("record_data")
).limit(20).show(truncate=False)

# Calculate difference between expected and actual
df_missing = df_missing.withColumn(
    "missing_count",
    col("expected_count").cast("int") - col("actual_count").cast("int")
)

print("\n📊 Number of missing fields per row:")
df_missing.groupBy("missing_count").count().orderBy("missing_count").show()

print("\n📊 Missing Fields Distribution:")
df_missing.groupBy("missing_fields").count().orderBy("count", ascending=False).show(truncate=False)

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 33, Finished, Available, Finished, False)

MISSING FIELDS ERRORS

🔍 Sample Missing Field Errors:


SynapseWidget(Synapse.DataFrame, 4a5a9aec-8171-47d8-a99d-1cdfed5dcd1c)


📊 Missing Field Patterns:
+--------------+------------+--------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|expected_count|actual_count|missing_fields      |record_data                                                                                                                                                                                                                                                                                                                                              |
+--------------+------------+--------------------+---------------------------------------------------------------------------------------------------------------------------------

### Three error patterns:
    Pattern 1: ['dept'] missing (5 records)
    Record has department instead of dept
    Both represent the same department value but with different column names

    Pattern 2: ['f_name', 'l_name'] missing (5 records)
    Record has name object with first and last nested inside
    Expected flat structure with separate f_name and l_name fields

    Pattern 3: ['salary'] missing (4 records)
    Record has compensation object with monthly_salary nested inside
    Expected flat salary field

The data is complete - it's just nested differently. No data is actually missing, it's a structural mismatch.

### Solutions:
    - For dept → department: Extract value from department and assign to dept column
    - For f_name/l_name → name: Extract first and last from the name object and assign to f_name and l_name
    - For salary → compensation: Extract monthly_salary from the compensation object and assign to salary

# 3: Solving the missing columns issues (Also under the hood it is the extra columns issues)

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
import json

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("FixJSONErrors").getOrCreate()

# Read error data
df_json_errors = spark.table("ODS.employee_json_data_errors")

current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Parse and fix records
fixed_records = []

for row in df_json_errors.collect():
    record_data = eval(row['record_data'])
    
    # Solution 1: dept → department
    if 'department' in record_data:
        record_data['dept'] = record_data['department']
        del record_data['department']
    
    # Solution 2: f_name/l_name → name
    if 'name' in record_data:
        if isinstance(record_data['name'], dict):
            record_data['f_name'] = record_data['name'].get('first')
            record_data['l_name'] = record_data['name'].get('last')
        del record_data['name']
    
    # Solution 3: salary → compensation
    if 'compensation' in record_data:
        if isinstance(record_data['compensation'], dict):
            record_data['salary'] = record_data['compensation'].get('monthly_salary')
        del record_data['compensation']
    
    # Add lineage columns
    record_data['load_timestamp'] = current_timestamp
    record_data['source_location'] = 'Tables/ODS/employee_json_data_errors'
    record_data['file_type'] = row['file_type']
    record_data['was_error'] = 'BOTH_MISSING_AND_EXTRA_FIXED'
    
    fixed_records.append(record_data)

# Define schema with lineage columns
schema = StructType([
    StructField("id", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("f_name", StringType(), True),
    StructField("l_name", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("dept", StringType(), True),
    StructField("country", StringType(), True),
    StructField("salary", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("employment_status", StringType(), True),
    StructField("education_level", StringType(), True),
    StructField("load_timestamp", StringType(), True),
    StructField("source_location", StringType(), True),
    StructField("file_type", StringType(), True),
    StructField("was_error", StringType(), True)
])

# Create DataFrame
df_json_fixed = spark.createDataFrame(fixed_records, schema)

print(f"✅ Fixed {df_json_fixed.count()} JSON error records")
print(f"Columns: {df_json_fixed.columns}")
display(df_json_fixed.limit(5).toPandas())

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 40, Finished, Available, Finished, False)

✅ Fixed 14 JSON error records
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type', 'was_error']


SynapseWidget(Synapse.DataFrame, 4dd68948-03be-41af-af36-48a7258ab2eb)

# 4: Investigate Extra Columns Errors

In [21]:
print("="*50)
print("EXTRA FIELDS ERRORS")
print("="*50)

# Show sample of extra field errors
print("\n🔍 Sample Extra Field Errors:")
display(df_extra.limit(10).toPandas())

# Analyze extra field patterns
print("\n📊 Extra Field Patterns:")
df_extra.select(
    col("expected_count"),
    col("actual_count"),
    col("extra_fields"),
    col("record_data")
).limit(20).show(truncate=False)

df_extra = df_extra.withColumn(
    "extra_count",
    col("actual_count").cast("int") - col("expected_count").cast("int")
)

print("\n📊 Number of extra fields per row:")
df_extra.groupBy("extra_count").count().orderBy("extra_count").show()

print("\n📊 Extra Fields Distribution:")
df_extra.groupBy("extra_fields").count().orderBy("count", ascending=False).show(truncate=False)

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 41, Finished, Available, Finished, False)

EXTRA FIELDS ERRORS

🔍 Sample Extra Field Errors:


SynapseWidget(Synapse.DataFrame, 44884f17-bd67-435c-8942-af2996103510)


📊 Extra Field Patterns:
+--------------+------------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|expected_count|actual_count|extra_fields    |record_data                                                                                                                                                                                                                                                                                                                                              |
+--------------+------------+----------------+-----------------------------------------------------------------------------------------------------------------------------------------------

### Already solved these issues above

# 5: General Exploration for the data frame that we should combine

In [25]:
df_json_fixed.head()

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 60, Finished, Available, Finished, False)

Row(id='100675', full_name='Layla Mahmoud', f_name='Layla', l_name='Mahmoud', dob='03-14-1969', dept='Information Technology', country='India', salary='$18,924.25', gender=' f ', job_title='Business Analyst', employment_status=' Contractor ', education_level='Diploma', load_timestamp='2026-07-18 12:41:46', source_location='Tables/ODS/employee_json_data_errors', file_type='json', was_error='BOTH _MISSING_AND_EXTRA_FIXED')

In [26]:
df_json_good.head()

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 63, Finished, Available, Finished, False)

Row(id='100624', full_name='James Williams', f_name='James', l_name='Williams', dob='03/10/1977', dept='Environmental', country='philippines', salary='37416.00', gender='female', job_title='Production Operator', employment_status='PROB', education_level='phd', load_timestamp='2026-07-17 20:17:34', source_location='Files/employee_dirty_files_pre_ODS/employee_dirty_json.json', file_type='json')

In [27]:
print("="*50)
print("JSON DATA RECONCILIATION")
print("="*50)

# Read JSON error data
df_json_errors = spark.table("ODS.employee_json_data_errors")

print("\n📊 Error Type Breakdown:")
df_json_errors.groupBy("error_type").count().show()

print("\n🔍 Let's examine the error records:")
display(df_json_errors.select("record_number", "error_type", "actual_count", "missing_fields", "extra_fields").toPandas())

print("\n" + "="*50)
print("FIXED RECORDS BREAKDOWN")
print("="*50)

print(f"df_json_good count: {df_json_good.count()} (Original good records)")
print(f"df_json_fixed count: {df_json_fixed.count()} (Fixed error records)")

print("\n📊 Combined fixed records:")
print(f"Total fixed: {df_json_fixed.count()}")
print(f"Should equal total errors: {df_json_errors.count()}")
print(f"Difference: {df_json_errors.count() - df_json_fixed.count()}")

print("\n" + "="*50)
print("FINAL RECONCILIATION")
print("="*50)

print(f"Original good: {df_json_good.count()}")
print(f"Fixed records: {df_json_fixed.count()}")
print(f"Total combined: {df_json_good.count() + df_json_fixed.count()}")
print(f"Expected total: {df_json_good.count() + df_json_errors.count()}")

if df_json_good.count() + df_json_fixed.count() == df_json_good.count() + df_json_errors.count():
    print("✅ NUMBERS MATCH!")
else:
    print("❌ NUMBERS DON'T MATCH!")

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 65, Finished, Available, Finished, False)

JSON DATA RECONCILIATION

📊 Error Type Breakdown:
+--------------------+-----+
|          error_type|count|
+--------------------+-----+
|MISSING_FIELDS | ...|   14|
+--------------------+-----+


🔍 Let's examine the error records:


SynapseWidget(Synapse.DataFrame, 755eee9c-acf2-4237-a07f-ebc93c8bbe9f)


FIXED RECORDS BREAKDOWN
df_json_good count: 986 (Original good records)
df_json_fixed count: 14 (Fixed error records)

📊 Combined fixed records:
Total fixed: 14
Should equal total errors: 14
Difference: 0

FINAL RECONCILIATION
Original good: 986
Fixed records: 14
Total combined: 1000
Expected total: 1000
✅ NUMBERS MATCH!


# 6: Merging the fixed rows with the already good rows

In [28]:
from pyspark.sql.functions import lit

# Add was_error column to df_json_good
df_json_good_with_flag = df_json_good.withColumn("was_error", lit("NO SCHEMA ERROR"))

# Union both dataframes
df_json_combined = df_json_good_with_flag.unionByName(df_json_fixed, allowMissingColumns=True)

print(f"✅ Combined all JSON records: {df_json_combined.count()}")
print(f"   Good records: {df_json_good.count()}")
print(f"   Fixed records: {df_json_fixed.count()}")

print("\n📊 Data quality breakdown:")
df_json_combined.groupBy("was_error").count().show()

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 66, Finished, Available, Finished, False)

✅ Combined all JSON records: 1000
   Good records: 986
   Fixed records: 14

📊 Data quality breakdown:
+--------------------+-----+
|           was_error|count|
+--------------------+-----+
|     NO SCHEMA ERROR|  986|
|BOTH _MISSING_AND...|   14|
+--------------------+-----+



# 7: Saving the combined data to the ODS layer

In [29]:
# Save df_json_combined to ODS schema in lakehouse
df_json_combined.write.format("delta").mode("overwrite").saveAsTable("ODS.employee_json_data_combined")

print(f"✅ Saved to ODS.employee_json_data_combined")
print(f"Total records: {df_json_combined.count()}")
print(f"Columns: {df_json_combined.columns}")

print("\n📊 Data quality breakdown:")
df_json_combined.groupBy("was_error").count().show()

print("\n📊 Sample data:")
display(df_json_combined.limit(5).toPandas())

StatementMeta(, bbcba6e2-ed8f-4dc2-92f4-1cf319aa234c, 67, Finished, Available, Finished, False)

✅ Saved to ODS.employee_json_data_combined
Total records: 1000
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type', 'was_error']

📊 Data quality breakdown:
+--------------------+-----+
|           was_error|count|
+--------------------+-----+
|     NO SCHEMA ERROR|  986|
|BOTH _MISSING_AND...|   14|
+--------------------+-----+


📊 Sample data:


SynapseWidget(Synapse.DataFrame, faa15014-2e08-4e48-9889-4794552a1643)